In [1]:
!pip install langchain_openai langchain_chroma langchain_community load_dotenv faiss-cpu sentence-transformers pymupdf GitPython

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.7/84.7 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 476.4/476.4 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 56.0 MB/s eta 0:00:00

In [2]:
from dotenv import load_dotenv
load_dotenv('./drive/MyDrive/새싹_딥러닝/api_keys.txt')

True

# Tavily

In [3]:
from langchain_community.tools.tavily_search import TavilySearchResults

tools = [TavilySearchResults(max_results=3)]

/tmp/ipython-input-2518516040.py:3: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tools = [TavilySearchResults(max_results=3)]


In [4]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

llm = ChatOpenAI(model='gpt-4o')
agent_executor = create_agent(llm, tools)

In [5]:
resp = agent_executor.invoke({'messages':[('user', 'LangChain의 개요를 알려줘.')]})

In [7]:
resp['messages'][-1].content

'LangChain는 개발자들이 대형 언어 모델(LLM)을 실제 애플리케이션에 통합할 수 있도록 돕는 오픈 소스 프레임워크입니다. 2022년 10월 Harrison Chase에 의해 시작되었으며, ChatGPT의 공개 런칭과 맞물려 AI 개발 커뮤니티에서 급속한 인기를 얻게 되었습니다. 2023년 중반에는 GitHub에서 가장 빠르게 성장하는 오픈 소스 프로젝트로 인정받았습니다. LangChain은 AI 기반 애플리케이션을 빠르고 효율적으로, 그리고 대규모로 구축할 수 있는 도구와 표준 인터페이스를 제공합니다.\n\nLangChain은 복잡하고 커스터마이즈된 체인을 쉽게 만들 수 있도록 도와주는 LangChain Expression Language (LCEL)과 같은 기능을 추가하여 지속적으로 발전하고 있고, 이를 통해 많은 변형과 커스터마이즈가 요구되는 초기의 생성 AI(Generative AI) 여행을 쉽게 만들어주고 있습니다.\n\n현재 LangChain은 100,000개 이상의 회사에서 사용하고 있으며, AI 애플리케이션 수명주기를 지원하는 플랫폼에서도 중점을 두고 있습니다. LangChain은 생산 단계의 애플리케이션 지원을 위해 LangSmith(모니터링 및 디버깅 도구)와 LangGraph(배포된 에이전트 오케스트레이션 도구) 같은 동반 도구도 도입하고 있습니다.\n\n하지만 2023년에는 빠른 개발 속도로 인해 자주 호환성 문제가 발생하였고, 릴리즈 노트의 명확한 안내 없이 업데이트가 갑자기 기존 코드를 깨뜨리는 경우가 많아 개발자들 사이에서 불만이 커지고 있습니다. 이러한 문제로 인해 개발자 커뮤니티에서는 LangChain의 오버헤드가 과연 가치가 있는지를 재검토하는 목소리가 증가했습니다.'

# FAISS

In [12]:
from langchain_community.document_loaders import GitLoader


def file_filter(file_path: str) -> bool:
    return file_path.endswith(".md")


loader = GitLoader(
    clone_url="https://github.com/langchain-ai/langchain",
    repo_path="./langchain",
    branch="master",
    file_filter=file_filter,
)

documents = loader.load()
print(len(documents))

36


In [13]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)

documents = text_splitter.split_documents(documents)
print(len(documents))

96


In [14]:
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_community.embeddings import HuggingFaceEmbeddings

In [15]:
embeddings_model = HuggingFaceEmbeddings(
    model_name='jhgan/ko-sbert-nli',
    model_kwargs={'device':'cpu'},
    encode_kwargs={'normalize_embeddings':True},
)


vectorstore = FAISS.from_documents(
    documents,
    embedding = embeddings_model,
    distance_strategy = DistanceStrategy.COSINE
)

/tmp/ipython-input-707567803.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/538 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [16]:
query = 'LangChain의 개요를 알려줘'

In [18]:
retriever = vectorstore.as_retriever(search_kwargs={'k':1})
docs = retriever.invoke(query)
print(len(docs))
print(docs[0])

1
page_content='LangChain is the easiest way to start building agents and applications powered by LLMs. With under 10 lines of code, you can connect to OpenAI, Anthropic, Google, and [more](https://docs.langchain.com/oss/python/integrations/providers/overview). LangChain provides a pre-built agent architecture and model integrations to help you get started quickly and seamlessly incorporate LLMs into your agents and applications.

We recommend you use LangChain if you want to quickly build agents and autonomous applications. Use [LangGraph](https://docs.langchain.com/oss/python/langgraph/overview), our low-level agent orchestration framework and runtime, when you have more advanced needs that require a combination of deterministic and agentic workflows, heavy customization, and carefully controlled latency.' metadata={'source': 'libs/langchain_v1/README.md', 'file_path': 'libs/langchain_v1/README.md', 'file_name': 'README.md', 'file_type': '.md'}


In [22]:
retriever = vectorstore.as_retriever(search_type='mmr', search_kwargs={'k':5, 'fetch_k':50})
docs = retriever.invoke(query)
print(len(docs))
print(docs[0])

5
page_content='LangChain is the easiest way to start building agents and applications powered by LLMs. With under 10 lines of code, you can connect to OpenAI, Anthropic, Google, and [more](https://docs.langchain.com/oss/python/integrations/providers/overview). LangChain provides a pre-built agent architecture and model integrations to help you get started quickly and seamlessly incorporate LLMs into your agents and applications.

We recommend you use LangChain if you want to quickly build agents and autonomous applications. Use [LangGraph](https://docs.langchain.com/oss/python/langgraph/overview), our low-level agent orchestration framework and runtime, when you have more advanced needs that require a combination of deterministic and agentic workflows, heavy customization, and carefully controlled latency.' metadata={'source': 'libs/langchain_v1/README.md', 'file_path': 'libs/langchain_v1/README.md', 'file_name': 'README.md', 'file_type': '.md'}


In [23]:
retriever = vectorstore.as_retriever(search_type='mmr', search_kwargs={'k':5, 'lambda_mult':0.15})
docs = retriever.invoke(query)
print(len(docs))
print(docs[-1])

5
page_content='- **Core layer** (`langchain-core`): Base abstractions, interfaces, and protocols. Users should not need to know about this layer directly.
- **Implementation layer** (`langchain`): Concrete implementations and high-level public utilities
- **Integration layer** (`partners/`): Third-party service integrations. Note that this monorepo is not exhaustive of all LangChain integrations; some are maintained in separate repos, such as `langchain-ai/langchain-google` and `langchain-ai/langchain-aws`. Usually these repos are cloned at the same level as this monorepo, so if needed, you can refer to their code directly by navigating to `../langchain-google/` from this monorepo.
- **Testing layer** (`standard-tests/`): Standardized integration tests for partner integrations

### Development tools & commands**' metadata={'source': 'AGENTS.md', 'file_path': 'AGENTS.md', 'file_name': 'AGENTS.md', 'file_type': '.md'}


In [24]:
# 유사도(cos) 값은 -1~1 사이의 값.
# threshold 값은 0~1 사이의 값.
# score_threshold 값이 커질수록 거의 동일한 문서만 허용. 작아질수록 관련없는 문서도 유입할 수 있음.
retriever = vectorstore.as_retriever(search_type='similarity_score_threshold', search_kwargs={'score_threshold':0.1})
docs = retriever.invoke(query)
print(len(docs))
print(docs)

4
[Document(id='5770faf3-561b-4299-85f7-f3fefdde241c', metadata={'source': 'libs/langchain_v1/README.md', 'file_path': 'libs/langchain_v1/README.md', 'file_name': 'README.md', 'file_type': '.md'}, page_content='LangChain is the easiest way to start building agents and applications powered by LLMs. With under 10 lines of code, you can connect to OpenAI, Anthropic, Google, and [more](https://docs.langchain.com/oss/python/integrations/providers/overview). LangChain provides a pre-built agent architecture and model integrations to help you get started quickly and seamlessly incorporate LLMs into your agents and applications.\n\nWe recommend you use LangChain if you want to quickly build agents and autonomous applications. Use [LangGraph](https://docs.langchain.com/oss/python/langgraph/overview), our low-level agent orchestration framework and runtime, when you have more advanced needs that require a combination of deterministic and agentic workflows, heavy customization, and carefully cont

In [26]:
result = vectorstore.similarity_search_with_score(query=query, k=4)

In [28]:
for doc, score in result:
  print(score)

0.73375213
0.7413166
0.7413166
0.7691163


In [29]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


# Retrieval
retriever = vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 5, 'lambda_mult': 0.15}
)

docs = retriever.invoke(query)

# Prompt
template = '''Answer the question based only on the following context:
{context}

Question: {question}
'''

prompt = ChatPromptTemplate.from_template(template)

# Model
llm = ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0,
    max_tokens=500,
)


def format_docs(docs):
    return '\n\n'.join([d.page_content for d in docs])

# Chain
chain = prompt | llm | StrOutputParser()

# Run
response = chain.invoke({'context': (format_docs(docs)), 'question':query})
response

'LangChain은 LLM(대형 언어 모델)으로 구동되는 에이전트 및 애플리케이션을 쉽게 구축할 수 있는 방법입니다. 10줄도 안 되는 코드로 OpenAI, Anthropic, Google 등과 연결할 수 있으며, 사전 구축된 에이전트 아키텍처와 모델 통합을 제공하여 빠르게 시작하고 LLM을 애플리케이션에 원활하게 통합할 수 있도록 돕습니다. LangChain을 사용하면 에이전트와 자율 애플리케이션을 신속하게 구축할 수 있으며, 더 복잡한 요구 사항이 있는 경우에는 LangGraph라는 저수준 에이전트 오케스트레이션 프레임워크를 사용할 수 있습니다.'

# Chroma

Chroma 활용한 전체 RAG 구조 한번에 쓴 코드

In [31]:
from langchain_community.document_loaders import GitLoader


def file_filter(file_path: str) -> bool:
    return file_path.endswith(".md")


loader = GitLoader(
    clone_url="https://github.com/langchain-ai/langchain",
    repo_path="./langchain",
    branch="master",
    file_filter=file_filter,
)

documents = loader.load()
print(len(documents))

from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)

docs = text_splitter.split_documents(documents)

from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

from langchain_chroma import Chroma

db = Chroma.from_documents(docs, embeddings)

retriever = db.as_retriever()



# 가장 유사도가 높은 문장을 하나만 추출
retriever = db.as_retriever(search_kwargs={'k': 1})

docs = retriever.invoke(query)
print(len(docs))
print(docs[0])


# MMR - 다양성 고려 (lambda_mult = 0.5)
retriever = db.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 5, 'fetch_k': 50}
)

docs = retriever.invoke(query)
print(len(docs))
docs[0]


from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


# Retrieval
retriever = db.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 5, 'lambda_mult': 0.15}
)

docs = retriever.invoke(query)

# Prompt
template = '''Answer the question based only on the following context:
{context}

Question: {question}
'''

prompt = ChatPromptTemplate.from_template(template)

# Model
llm = ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0,
    max_tokens=500,
)


def format_docs(docs):
    return '\n\n'.join([d.page_content for d in docs])

# Chain
chain = prompt | llm | StrOutputParser()

# Run
response = chain.invoke({'context': (format_docs(docs)), 'question':query})
response

36
1
page_content='LangChain is a framework for building agents and LLM-powered applications. It helps you chain together interoperable components and third-party integrations to simplify AI application development – all while future-proofing decisions as the underlying technology evolves.

```bash
pip install langchain
```

If you're looking for more advanced customization or agent orchestration, check out [LangGraph](https://docs.langchain.com/oss/python/langgraph/overview), our framework for building controllable agent workflows.

---

**Documentation**:

- [docs.langchain.com](https://docs.langchain.com/oss/python/langchain/overview) – Comprehensive documentation, including conceptual overviews and guides
- [reference.langchain.com/python](https://reference.langchain.com/python) – API reference docs for LangChain packages

**Discussions**: Visit the [LangChain Forum](https://forum.langchain.com) to connect with the community and share all of your technical questions, ideas, and fee

'LangChain은 에이전트 및 LLM 기반 애플리케이션을 구축하기 위한 프레임워크입니다. 이 프레임워크는 상호 운용 가능한 구성 요소와 제3자 통합을 연결하여 AI 애플리케이션 개발을 간소화하며, 기술이 발전함에 따라 미래에 대비할 수 있는 결정을 내릴 수 있도록 돕습니다. LangChain은 모듈화, 안정성, 그리고 검증된 성능을 제공하는 `langchain-core`를 기반으로 하여, 다양한 모델 제공자가 요구하는 인터페이스를 구현할 수 있도록 설계되었습니다. 이를 통해 LangChain 생태계의 나머지 부분에서 쉽게 사용할 수 있습니다.'

In [35]:
# split
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)

docs = text_splitter.split_documents(documents)

In [36]:
# embedding
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

In [37]:
# Chroma로 db 생성
from langchain_chroma import Chroma

db = Chroma.from_documents(docs, embeddings)

In [38]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [39]:
# Retrieval
retriever = db.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 5, 'lambda_mult': 0.15}
)

docs = retriever.invoke(query)

In [40]:
# Prompt
template = '''Answer the question based only on the following context:
{context}

Question: {question}
'''

prompt = ChatPromptTemplate.from_template(template)

In [41]:
# Model
llm = ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0,
    max_tokens=500,
)

In [42]:
query = 'LangChain의 개요를 알려줘'

In [43]:
def format_docs(docs):
    return '\n\n'.join([d.page_content for d in docs])

# Chain
chain = prompt | llm | StrOutputParser()

# Run
response = chain.invoke({'context': (format_docs(docs)), 'question':query})
response

'LangChain은 에이전트 및 LLM(대형 언어 모델) 기반 애플리케이션을 구축하기 위한 프레임워크입니다. 이 프레임워크는 상호 운용 가능한 구성 요소와 제3자 통합을 연결하여 AI 애플리케이션 개발을 간소화하며, 기술이 발전함에 따라 미래에 대비할 수 있는 결정을 내릴 수 있도록 돕습니다. LangChain은 모듈화된 아키텍처를 통해 개발자들이 다양한 데이터 소스와 시스템에 LLM을 쉽게 연결하고, 모델을 교체하며, 신속하게 프로토타입을 제작할 수 있도록 지원합니다. 또한, 안정적인 배포를 위한 기능과 활발한 커뮤니티 생태계를 제공합니다. LangChain은 다양한 통합, 템플릿 및 커뮤니티 기여 구성 요소를 활용하여 지속적인 개선과 최신 AI 개발 동향을 반영합니다.'

# Prompt 증강 - pydantic 사용

- agent나 LangGraph 만들 때 이 구조를 사용

In [53]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

In [54]:
openai_model: str = "gpt-4o"
temperature: float = 0.0

In [55]:
# BaseModel 때문에 class 형태가 일반적인 파이썬 class와 다름
# 일반적인 파이썬에서의 class는 __init__이 실행되고 그다음 속성이 실행됨
# description의 형태를 강제하는 것.
class Goal(BaseModel):
  description: str = Field(..., description="목표 설명")

# @property는 클래스 중에서 함수를 변수처럼 쓸 수 있게 해주는 decorator
# getter, setter 대신에 쓰이는 것
  @property
  def text(self) -> str:
    return f"{self.description}"

## Passive Goal Creator

- agent가 계속 검색을 여러 번하다가 도구를 잘못 쓰고 중간에 방향을 잃는데, 이를 해결해주기 위해 Goal을 고정해줘야함.

- query -> 일회성
- goal -> 지속되는 목적

In [61]:
class PassiveGoalCreator:
  def __init__(self, llm: ChatOpenAI,):
    self.llm = llm

  def run(self, query:str) -> Goal:
      prompt = ChatPromptTemplate.from_template(
            "사용자 입력을 분석하여 명확하고 실행 가능한 목표를 생성해 주세요.\n"
            "요건:\n"
            "1. 목표는 구체적이고 명확해야 하며, 실행 가능한 수준으로 상세화되어야 합니다.\n"
            "2. 당신이 실행할 수 있는 행동은 다음과 같은 행동뿐입니다.\n"
            "   - 인터넷을 이용하여 목표 달성을 위한 조사를 수행합니다.\n"
            "   - 사용자를 위한 보고서를 생성합니다.\n"
            "3. 절대 2.에 명시된 행동 외의 다른 행동을 취해서는 안 됩니다.\n"
            "사용자 입력: {query}"
        )
      chain = prompt | self.llm.with_structured_output(Goal)
      return chain.invoke({'query':query})

In [62]:
task = '삼성전자 주식에 대해 평가해줘'

In [63]:
llm = ChatOpenAI(model=openai_model, temperature=temperature)
goal_creator = PassiveGoalCreator(llm=llm)
result: Goal = goal_creator.run(query=task)
print(f"{result.text}")

### 목표: 삼성전자 주식 평가 보고서 작성

#### 1. 목표 설명
삼성전자 주식에 대한 포괄적이고 명확한 평가 보고서를 작성합니다. 이 보고서는 사용자가 삼성전자 주식에 대한 투자 결정을 내리는 데 도움을 줄 수 있도록 설계됩니다.

#### 2. 목표 달성을 위한 세부 단계

1. **삼성전자 주식의 현재 시장 상황 조사**
   - 인터넷을 통해 삼성전자 주식의 최신 주가, 거래량, 시가총액 등의 정보를 수집합니다.
   - 최근 주가 변동 추세와 관련된 뉴스 및 분석 자료를 조사합니다.

2. **삼성전자의 재무 상태 분석**
   - 삼성전자의 최근 분기 및 연간 재무제표를 검토하여 수익성, 유동성, 부채 비율 등을 분석합니다.
   - 주요 재무 지표(예: PER, PBR, ROE 등)를 조사하여 경쟁사와 비교합니다.

3. **산업 및 시장 동향 분석**
   - 반도체 및 전자 산업의 최신 동향과 삼성전자의 시장 내 위치를 조사합니다.
   - 글로벌 경제 상황이 삼성전자 주식에 미치는 영향을 분석합니다.

4. **전문가 의견 및 투자자 심리 조사**
   - 금융 전문가 및 애널리스트의 삼성전자 주식에 대한 평가 및 추천 의견을 수집합니다.
   - 투자자 포럼 및 커뮤니티에서의 삼성전자 주식에 대한 일반적인 투자자 심리를 조사합니다.

5. **보고서 작성 및 제공**
   - 수집한 데이터를 바탕으로 삼성전자 주식의 강점, 약점, 기회, 위협(SWOT 분석)을 포함한 종합 평가 보고서를 작성합니다.
   - 보고서를 사용자에게 제공하여 삼성전자 주식에 대한 명확한 투자 결정을 내릴 수 있도록 지원합니다.

#### 3. 목표 달성의 기대 효과
- 사용자는 삼성전자 주식에 대한 명확하고 객관적인 정보를 바탕으로 투자 결정을 내릴 수 있습니다.
- 보고서는 사용자가 주식 시장의 복잡성을 이해하고, 보다 전략적인 투자 접근 방식을 취하는 데 도움을 줄 것입니다.


### 측정 기준 추가

In [70]:
class OptimizedGoal (BaseModel):
  description: str = Field(..., description="목표 설명")
  metrics: str = Field(..., description="목표의 달성도를 측정하는 방법")

  @property
  def text(self) -> str:
    return f"{self.description}(\n\n측정 기준:{self.metrics})"

In [71]:
class PromptOptimizer:
  def __init__(self, llm: ChatOpenAI,):
    self.llm = llm

  def run(self, query:str) -> Goal:
      prompt = ChatPromptTemplate.from_template(
            "당신은 목표 설정 전문가입니다. 아래의 목표를 SMART 원칙(Specific: 구체적, Measurable: 측정 가능, Achievable: 달성 가능, Relevant: 관련성이 높은, Time-bound: 기한이 있는)에 기반하여 최적화해 주세요.\n\n"
            "원래 목표:\n"
            "{query}\n\n"
            "지시 사항:\n"
            "1. 원래 목표를 분석하고, 부족한 요소나 개선점을 파악해 주세요.\n"
            "2. 당신이 실행할 수 있는 행동은 다음과 같습니다.\n"
            "   - 인터넷을 이용하여 목표 달성을 위한 조사를 수행한다.\n"
            "   - 사용자를 위한 보고서를 생성한다.\n"
            "3. SMART 원칙의 각 요소를 고려하면서 목표를 구체적이고 상세하게 기술해 주세요.\n"
            "   - 절대 추상적인 표현을 포함해서는 안 됩니다.\n"
            "   - 반드시 모든 단어가 실행 가능하고 구체적인지 확인해 주세요.\n"
            "4. 목표의 달성도를 측정하는 방법을 구체적이고 상세하게 기술해 주세요.\n"
            "5. 원래 목표에서 기한이 지정되지 않은 경우에는 기한을 고려할 필요가 없습니다.\n"
            "6. 주의: 절대로 2번 이외의 행동을 취해서는 안 됩니다."
        )
      chain = prompt | self.llm.with_structured_output(OptimizedGoal)
      return chain.invoke({'query':query})

In [72]:
llm = ChatOpenAI(model=openai_model, temperature=temperature)
passive_goal_creator = PassiveGoalCreator(llm=llm)
goal: Goal = passive_goal_creator.run(query=task)

prompt_optimizer = PromptOptimizer(llm=llm)
optimized_goal: OptimizedGoal = prompt_optimizer.run(query=goal.text)
print(f"{optimized_goal.text}")

### 최적화된 목표: 삼성전자 주식 평가 보고서 작성

#### 1. 목표 설명
삼성전자 주식에 대한 포괄적이고 명확한 평가 보고서를 작성하여, 사용자가 삼성전자 주식의 현재 상태와 미래 전망을 이해하고 투자 결정을 내리는 데 도움을 줍니다. 이 보고서는 삼성전자의 최근 주가 변동, 시장 동향, 경쟁사 비교, 그리고 전문가 의견을 포함합니다.

#### 2. 목표 달성을 위한 세부 단계

1. **삼성전자 주식의 현재 상태 조사**
   - 삼성전자의 최근 6개월간 주가 변동 및 거래량 분석
   - 최근 2분기 실적 및 재무제표 검토
   - 지난 3개월간 주요 뉴스 및 발표 사항 확인

2. **시장 동향 및 경쟁사 분석**
   - 최근 1년간 반도체 및 전자 산업의 트렌드 조사
   - 주요 경쟁사(TSMC, 인텔 등)와의 최근 6개월간 비교 분석
   - 글로벌 경제 상황이 삼성전자에 미치는 영향 평가

3. **미래 전망 및 전문가 의견 수집**
   - 삼성전자의 향후 1년간 기술 개발 및 사업 전략 조사
   - 주식 시장 전문가 및 애널리스트의 최근 3개월간 의견 및 예측 수집
   - 장기적인 투자 관점에서의 삼성전자 주식 평가

4. **보고서 작성 및 제공**
   - 수집한 데이터를 바탕으로 20페이지 내외의 명확하고 체계적인 보고서 작성
   - 사용자가 이해하기 쉽게 5개 이상의 그래프 및 표를 활용하여 시각화
   - 최종 보고서를 PDF 형식으로 사용자에게 제공하여 삼성전자 주식에 대한 명확한 평가를 전달

#### 3. 목표 달성의 기대 효과
- 사용자는 삼성전자 주식에 대한 명확한 이해를 얻게 되며, 이를 바탕으로 보다 신중한 투자 결정을 내릴 수 있습니다.
- 보고서는 사용자가 삼성전자의 현재 위치와 미래 가능성을 평가하는 데 유용한 자료로 활용될 것입니다.(

측정 기준:- 보고서 작성 완료 여부
- 보고서 내 포함된 그래프 및 표의 수
- 보고서의 페이지 수
- 보고서 제공 후 사용자 피드백 수집 및 분석
- 보고서 제